# 08 — User 666 Hyperbolic Preparation (Steps 1–2)

This notebook implements only the first two setup steps for the hyperbolic pipeline:

1. Extract observed ratings of user 666.
2. Define a real-valued mapping `t = f(r)` from ratings.

> Note: Hyperbolic-state mapping (`z = cosh(t) + u sinh(t)`) is intentionally **not** included yet.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

rating_path = root / "data" / "rating.csv"
assert rating_path.exists(), f"Not found: {rating_path}"

df = pd.read_csv(
    rating_path,
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32"},
    parse_dates=["timestamp"],
)

print(f"Loaded ratings: {len(df):,}")
print(f"Users: {df['userId'].nunique():,} | Movies: {df['movieId'].nunique():,}")

Loaded ratings: 20,000,263
Users: 138,493 | Movies: 26,744


## Step 1 — Extract observed ratings of user 666

In [2]:
UID = 666
u666 = df[df["userId"] == UID].copy().sort_values("movieId").reset_index(drop=True)
assert not u666.empty, "User 666 not found in rating.csv"

print(f"User {UID} observed ratings: {len(u666)}")
display(u666[["userId", "movieId", "rating", "timestamp"]].head(10))

User 666 observed ratings: 68


,userId,movieId,rating,timestamp
0,666,1,5.0,1997-10-07 18:27:41
1,666,6,4.0,1997-10-07 18:30:15
2,666,17,5.0,1997-10-07 18:25:41
3,666,25,5.0,1997-10-07 18:30:15
4,666,32,4.0,1997-10-07 18:31:08
5,666,36,4.0,1997-10-07 18:25:42
6,666,41,2.0,1997-10-07 18:29:45
7,666,50,4.0,1997-10-07 18:27:06
8,666,52,3.0,1997-10-07 18:25:02
9,666,62,5.0,1997-10-07 18:31:55


## Step 2 — Define and compute real mapping `t = f(r)`

Use a simple centered-and-scaled mapping:

$$t = \alpha (r - \mu)$$

- `r`: observed rating
- `mu`: reference mean (here: user-666 mean)
- `alpha`: scale factor

In [3]:
alpha = 0.8
mu = float(u666["rating"].mean())

u666["t"] = alpha * (u666["rating"].astype(float) - mu)

print(f"Mapping setup: t = alpha * (r - mu)")
print(f"alpha = {alpha:.3f}, mu (user-666 mean) = {mu:.6f}")
print(f"t stats: min={u666['t'].min():.6f}, max={u666['t'].max():.6f}, mean={u666['t'].mean():.6f}")

preview = u666[["movieId", "rating", "t"]].copy()
preview["rating"] = preview["rating"].round(3)
preview["t"] = preview["t"].round(6)
display(preview.head(15))

Mapping setup: t = alpha * (r - mu)
alpha = 0.800, mu (user-666 mean) = 3.897059
t stats: min=-2.317647, max=0.882353, mean=0.000000


,movieId,rating,t
0,1,5.0,0.882353
1,6,4.0,0.082353
2,17,5.0,0.882353
3,25,5.0,0.882353
4,32,4.0,0.082353
5,36,4.0,0.082353
6,41,2.0,-1.517647
7,50,4.0,0.082353
8,52,3.0,-0.717647
9,62,5.0,0.882353


## Step 3 — Map `t` to hyperbolic state

Use the hyperbolic-number embedding (unit-hyperbola parameterization):

$$
z = \cosh(t) + u\sinh(t)
$$

We store the real and hyperbolic components as:
- `x = cosh(t)`
- `y = sinh(t)`

So each observed rating of user 666 gets one hyperbolic-state pair `(x, y)`.

In [4]:
# Step 3 — Compute hyperbolic state components from t

assert "t" in u666.columns, "Run Step 2 first to compute t."

u666["x"] = np.cosh(u666["t"].astype(float))
u666["y"] = np.sinh(u666["t"].astype(float))

# Consistency check for unit-hyperbola parameterization:
# cosh(t)^2 - sinh(t)^2 = 1
u666["x2_minus_y2"] = u666["x"]**2 - u666["y"]**2

print("Step 3 completed: hyperbolic mapping z = cosh(t) + u*sinh(t)")
print(
    "x2-y2 stats: "
    f"min={u666['x2_minus_y2'].min():.8f}, "
    f"max={u666['x2_minus_y2'].max():.8f}, "
    f"mean={u666['x2_minus_y2'].mean():.8f}"
)

hyper_preview = u666[["movieId", "rating", "t", "x", "y", "x2_minus_y2"]].copy()
hyper_preview[["rating", "t", "x", "y", "x2_minus_y2"]] = hyper_preview[["rating", "t", "x", "y", "x2_minus_y2"]].round(6)
display(hyper_preview.head(15))

Step 3 completed: hyperbolic mapping z = cosh(t) + u*sinh(t)
x2-y2 stats: min=1.00000000, max=1.00000000, mean=1.00000000


,movieId,rating,t,x,y,x2_minus_y2
0,1,5.0,0.882353,1.415194,1.001386,1.0
1,6,4.0,0.082353,1.003393,0.082446,1.0
2,17,5.0,0.882353,1.415194,1.001386,1.0
3,25,5.0,0.882353,1.415194,1.001386,1.0
4,32,4.0,0.082353,1.003393,0.082446,1.0
5,36,4.0,0.082353,1.003393,0.082446,1.0
6,41,2.0,-1.517647,2.390353,-2.171126,1.0
7,50,4.0,0.082353,1.003393,0.082446,1.0
8,52,3.0,-0.717647,1.268752,-0.780853,1.0
9,62,5.0,0.882353,1.415194,1.001386,1.0
